# Hipótese: Antecedência do cancelamento x Taxa de detratores

Testa se o intervalo registrado em `ANTECEDENCIA_CANCELAMENTO` está associado à taxa de passageiros detratores (`NPS_PRINCIPAL == -100`).

> **Atenção à unidade:** o dicionário de dados descreve o intervalo em dias, enquanto a interpretação anterior usava horas. Até a confirmação com o parceiro, as faixas e os efeitos abaixo são apresentados em **unidades registradas**, sem convertê-las em horas ou dias.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 150)

## 1. Carregar e unir as bases

In [ ]:
nps = pd.concat(
    [pd.read_csv(f'PROJETO_INTELI_NPS_0{i}.csv') for i in range(1, 5)],
    ignore_index=True
)

viagem = pd.read_csv('PROJETO_INTELI_INFORMACAO_VIAGEM.csv')

df = nps.merge(viagem, on='RESPONDENT_ID', how='inner')
duplicados = df['RESPONDENT_ID'].duplicated().sum()
df = df.drop_duplicates(subset='RESPONDENT_ID', keep='first').copy()
df['DETRATOR'] = df['NPS_PRINCIPAL'] == -100

print(f'Registros após união e deduplicação: {len(df):,}')
print(f'Respondent IDs duplicados removidos: {duplicados:,}')
print(f'Dimensões da base analítica: {df.shape}')

## 2. Isolar voos cancelados e comparar com voos não cancelados

In [ ]:
taxa_geral = df.groupby('CANCELAMENTO_VOO')['DETRATOR'].mean()
print(taxa_geral)

## 3. Segmentar cancelamentos por faixa de antecedência

As faixas abaixo usam os valores brutos da variável. A unidade de medida ainda precisa ser confirmada antes de interpretá-las como horas ou dias.

In [ ]:
cancelados = df[df['CANCELAMENTO_VOO'] == True].copy()

bins = [-1, 0, 6, 24, 48, np.inf]
labels = ['0 unidades', '1-6 unidades', '7-24 unidades', '25-48 unidades', '>48 unidades']
cancelados['FAIXA_ANTECEDENCIA'] = pd.cut(
    cancelados['ANTECEDENCIA_CANCELAMENTO'], bins=bins, labels=labels
)

tabela = cancelados.groupby('FAIXA_ANTECEDENCIA', observed=False)['DETRATOR'].agg(
    pct_detrator='mean', n='size'
)
tabela.index.name = 'FAIXA_ANTECEDENCIA (unidade a confirmar)'
tabela

## 4. Visualizar o gradiente

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tabela['pct_detrator'].mul(100).plot(kind='bar', ax=ax, color='#1f77b4')

taxa_base = df[df['CANCELAMENTO_VOO'] == False]['DETRATOR'].mean() * 100
ax.axhline(taxa_base, color='red', linestyle='--',
           label=f'Base sem cancelamento ({taxa_base:.1f}%)')

ax.set_ylabel('% Detratores')
ax.set_xlabel('Antecedência registrada (unidade a confirmar)')
ax.set_title('Antecedência registrada x Taxa de detratores')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Teste estatístico de associação (qui-quadrado)

In [ ]:
tabela_contingencia = pd.crosstab(cancelados['FAIXA_ANTECEDENCIA'], cancelados['DETRATOR'])
chi2, p, dof, expected = stats.chi2_contingency(tabela_contingencia)

print(f'chi2 = {chi2:.2f}')
print(f'p-valor = {p:.2e}')
print(f'graus de liberdade = {dof}')

In [ ]:
corr = cancelados['ANTECEDENCIA_CANCELAMENTO'].corr(cancelados['NPS_PRINCIPAL'])
print(f'Correlação ANTECEDENCIA_CANCELAMENTO x NPS_PRINCIPAL: {corr:.3f}')

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

modelo_df = cancelados.dropna(subset=['ANTECEDENCIA_CANCELAMENTO', 'DETRATOR']).copy()
modelo_df['DETRATOR'] = modelo_df['DETRATOR'].astype(int)

modelo = smf.logit(
    'DETRATOR ~ ANTECEDENCIA_CANCELAMENTO',
    data=modelo_df
).fit(disp=False)

print(modelo.summary())
print()
print('Odds ratio por unidade adicional de antecedência:',
      np.exp(modelo.params['ANTECEDENCIA_CANCELAMENTO']))

## Conclusão

- Na escala original da variável, a taxa de detratores cai conforme aumenta a antecedência registrada (69,2% → 24,6%), aproximando-se da taxa base de voos sem cancelamento (18,2%).
- O teste qui-quadrado confirma associação estatisticamente significativa entre a faixa de antecedência registrada e a taxa de detratores.
- A regressão estima redução nas chances de detratar a cada unidade adicional registrada. Essa unidade não deve ser chamada de hora ou dia até a validação do dicionário com o parceiro; portanto, recomendações que dependam de uma antecedência específica permanecem pendentes dessa confirmação.